In [1]:
!pip install catboost xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.1 MB/s eta 0:00:00


In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [3]:
df = pd.read_csv("EMS_Feature_Engineered.csv")

print(df.shape)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'EMS_Feature_Engineered.csv'

In [20]:
target = "incident_response_seconds_qy"

In [21]:
drop_cols = [
    "incident_response_seconds_qy",
    "dispatch_response_seconds_qy",
    "incident_travel_tm_seconds_qy",
    "travel_duration",
    "hospital_time",
    "closure_time"
]

X = df.drop(columns=drop_cols)

y = df["incident_response_seconds_qy"]

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(656839, 19)
(164210, 19)


In [23]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [26]:
xgb = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",
    random_state=42
)
xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

In [27]:
cat = CatBoostRegressor(
    iterations=300,
    learning_rate=0.1,
    depth=8,
    random_state=42,
    verbose=100
)
cat.fit(X_train, y_train)

cat_pred = cat.predict(X_test)

0:	learn: 1050.3667957	total: 80.6ms	remaining: 24.1s
100:	learn: 681.6947877	total: 8.36s	remaining: 16.5s
200:	learn: 661.4579203	total: 16.1s	remaining: 7.94s
299:	learn: 648.7181443	total: 24.3s	remaining: 0us


In [28]:
def evaluate(name, y_true, pred):

    mae = mean_absolute_error(y_true, pred)

    rmse = np.sqrt(mean_squared_error(y_true, pred))

    r2 = r2_score(y_true, pred)

    print("="*50)

    print(name)

    print("MAE :", round(mae,2))

    print("RMSE :", round(rmse,2))

    print("R2 :", round(r2,4))

In [29]:
evaluate("Random Forest", y_test, rf_pred)

evaluate("CatBoost", y_test, cat_pred)

evaluate("XGBoost", y_test, xgb_pred)

Random Forest
MAE : 330.01
RMSE : 713.03
R2 : 0.5977
CatBoost
MAE : 318.74
RMSE : 686.17
R2 : 0.6274
XGBoost
MAE : 320.24
RMSE : 694.43
R2 : 0.6184


In [30]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": cat.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(20)

,Feature,Importance
11,dispatch_delay,34.370444
10,incident_duration,26.391724
5,hour,9.378508
1,final_call_type,3.894479
18,final_call_frequency,3.889830
0,initial_call_type,3.786130
2,zipcode,3.773786
6,month,2.935985
17,initial_call_frequency,2.861950
14,zipcode_frequency,1.757547
